<a href="https://colab.research.google.com/github/2303A51589/compilerdesign/blob/main/lexical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
%%writefile parser.y
%{
    #include<stdio.h>
    int yylex(void);
    void yyerror(char *s);
%}
%token a b

%%
s: a s b { printf("accepted\n");}
| a b    {printf("accepted\n");}
;

%%
void yyerror(char *s) {
    printf("given input is error\n");
}

Overwriting parser.y


In [13]:
 %%writefile lexer.l
%{
#include "y.tab.h"
%}

%%
[aA]    { return a; }
[bB]    { return b; }
[ \t\n] { /* ignore whitespace */ }
.       { return yytext[0]; }
%%
int yywrap() { return 1; }

int main() {
    printf("Enter the string: ");
    yyparse();
    return 0;
}

Writing lexer.l


In [24]:
!bison -d parser.y -o y.tab.c
!flex -o lex.yy.c lexer.l
!gcc y.tab.c lex.yy.c -o parser

In [22]:
# Test the compiled parser with a valid input string
!echo "aaabbb" | ./parser

/bin/bash: line 1: ./parser: No such file or directory


In [23]:
# Test the compiled parser with an invalid input string
!echo "aaabb" | ./parser

/bin/bash: line 1: ./parser: No such file or directory


In [26]:
%%writefile expression_parser.y
%{
#include <stdio.h>
int yylex(void);
void yyerror(char *s);
%}

%token A

%%
E : E '+' E { printf("Valid Expression\n"); }
  | A       { }
  ;

%%
int main()
{
  printf("Enter an expression:\n");
  yyparse();
  return 0;
}

void yyerror(char *s)
{
  printf("Invalid Expression\n");
}

Writing expression_parser.y


In [28]:
%%writefile expression_lexer.l
%{
    #include "y.tab.h"
%}

%%
a   {return A;}
"+" {return '+';}
[\t\n] ;
.   {return yytext[0];}
%%

int yywrap()
{
  return 1;
}

Writing expression_lexer.l


In [33]:
%%writefile lexer.l
%{
#include "y.tab.h"
#include <stdlib.h>
void yyerror(const char *s);
%}

%%
[0-9]+  { yylval = atoi(yytext); return NUMBER; }
[ \t]   ; /* ignore whitespace */
\n      { return '\n'; }
.       { return yytext[0]; }

%%
int yywrap() {
    return 1;
}

Overwriting lexer.l


In [34]:
%%writefile parser.y
%{
#include <stdio.h>
#include <stdlib.h>
int yylex(void);
void yyerror(const char *s);
%}

%token NUMBER
%left '+' '-'
%left '*' '/'

%%
input:
    expr '\n' { printf("result = %d\n", $1); exit(0); }
    ;

expr:
    expr '+' expr { $$ = $1 + $3; }
    | expr '-' expr { $$ = $1 - $3; }
    | expr '*' expr { $$ = $1 * $3; }
    | expr '/' expr {
        if ($3 == 0) {
            yyerror("Division by zero");
            exit(1);
        } else {
            $$ = $1 / $3;
        }
    }
    | NUMBER { $$ = $1; }
    ;

%%
void yyerror(const char *s) {
    fprintf(stderr, "Error: %s\n", s);
}

int main() {
    printf("Enter an expression:\n");
    yyparse();
    return 0;
}

Overwriting parser.y


In [35]:
%%bash
# Compile parser.y and lexer.l
bison -d parser.y -o y.tab.c
flex -o lex.yy.c lexer.l

# Compile and link with gcc
gcc y.tab.c lex.yy.c -o calculator

# Run calculator test
echo "15+5*3" | ./calculator

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
/usr/bin/ld: /usr/lib/gcc/x86_64-linux-gnu/13/../../../x86_64-linux-gnu/Scrt1.o: in function `_start':
(.text+0x1b): undefined reference to `main'
collect2: error: ld returned 1 exit status
bash: line 12: ./calculator: No such file or directory


CalledProcessError: Command 'b'# Install required tools\napt-get update -qq && apt-get install -y -qq bison flex\n\n# Compile parser.y and lexer.l\nbison -d parser.y -o y.tab.c\nflex -o lex.yy.c lexer.l\n\n# Compile and link with gcc\ngcc y.tab.c lex.yy.c -o calculator\n\n# Run calculator test\necho "15+5*3" | ./calculator\n'' returned non-zero exit status 127.